# Chapter 5 Lab — Reasoning Patterns

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 5 — Reasoning Patterns  
**Goal:** Understand and implement the core reasoning strategies that transform basic LLM calls into reliable agentic workflows.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | When Direct Prompting Breaks |
| 3 | Chain-of-Thought Prompting |
| 4 | Tree-of-Thought Reasoning |
| 5 | Self-Consistency |
| 6 | Reflection and Self-Critique |
| 7 | Reasoning Improves Tool Selection |
| 8 | Choosing a Reasoning Strategy |
| 9 | Exercises |

> **Prerequisites:** An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`.  
> All sections include **simulated LLM responses** so the notebook is fully runnable without an API key.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install openai python-dotenv --quiet

In [ ]:
import json
import os
import re
import textwrap
import random
from collections import Counter
from typing import Optional

from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

# ── Toggle between live API calls and simulated responses ──────────────────
# Set USE_LIVE_API = True if you have an API key and want real responses.
USE_LIVE_API = False

if USE_LIVE_API:
    import openai
    client = openai.OpenAI()
    print("Mode: LIVE API calls")
else:
    client = None
    print("Mode: SIMULATED responses (no API key needed)")

In [ ]:
# ── Helper: call LLM or return simulated response ────────────────────────────

def call_llm(messages: list[dict],
             simulated_response: str,
             model: str = "gpt-4o-mini",
             temperature: float = 0.7) -> str:
    """Call the OpenAI API if USE_LIVE_API is True, otherwise return the
    simulated_response so the notebook runs without an API key."""
    if USE_LIVE_API and client is not None:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
        )
        return resp.choices[0].message.content
    return simulated_response


def print_box(title: str, content: str, width: int = 78):
    """Pretty-print a titled box for comparing outputs."""
    border = "=" * width
    print(f"\n{border}")
    print(f"  {title}")
    print(border)
    for line in content.strip().split("\n"):
        print(f"  {line}")
    print(border)

---
## 2. When Direct Prompting Breaks

Direct prompting means asking the LLM a question and taking whatever it returns at face value. This works fine for simple lookups but collapses on tasks that require **multi-step reasoning**, **arithmetic**, or **weighing trade-offs**.

Below we pose a logic puzzle that most LLMs get wrong with a single direct prompt.

In [ ]:
# ── A problem that stumps direct prompting ───────────────────────────────────

HARD_PROBLEM = """
A farmer has 3 fields. Field A produces 240 bushels of wheat per acre.
Field B produces 180 bushels per acre but costs 30% less to maintain.
Field C produces 300 bushels per acre but costs 60% more to maintain.

The base maintenance cost is $50 per acre. The farmer has a budget of $10,000
for maintenance and wants to maximize total bushels. Each field can use at most
40 acres. How should the farmer allocate acres to each field?
""".strip()

print("Problem:")
print(textwrap.fill(HARD_PROBLEM, width=78))

In [ ]:
# ── Direct prompting attempt ─────────────────────────────────────────────────

direct_prompt = [{"role": "user", "content": HARD_PROBLEM}]

SIMULATED_DIRECT = """The farmer should allocate:
- Field A: 40 acres (240 * 40 = 9,600 bushels)
- Field B: 40 acres (180 * 40 = 7,200 bushels)
- Field C: 40 acres (300 * 40 = 12,000 bushels)

Total: 28,800 bushels.

This maximizes output by using all available acreage."""

direct_answer = call_llm(direct_prompt, SIMULATED_DIRECT)
print_box("Direct Prompting Result", direct_answer)

In [ ]:
# ── Why is the direct answer wrong? ──────────────────────────────────────────
# Let's verify the budget constraint that the LLM ignored.

base_cost = 50  # $ per acre
costs = {
    "A": base_cost * 1.0,           # $50/acre
    "B": base_cost * 0.70,          # $35/acre (30% less)
    "C": base_cost * 1.60,          # $80/acre (60% more)
}
yields = {"A": 240, "B": 180, "C": 300}
budget = 10_000
max_acres = 40

# The direct answer allocated 40 acres to each field.
direct_alloc = {"A": 40, "B": 40, "C": 40}
direct_total_cost = sum(direct_alloc[f] * costs[f] for f in direct_alloc)
direct_total_bushels = sum(direct_alloc[f] * yields[f] for f in direct_alloc)

print(f"Direct answer total cost:    ${direct_total_cost:,.0f}")
print(f"Budget:                      ${budget:,}")
print(f"Over budget by:              ${direct_total_cost - budget:,.0f}")
print(f"\nThe direct answer VIOLATES the budget constraint.")
print(f"The LLM ignored maintenance costs entirely.")

**Takeaway:** Direct prompting fails here because the LLM needs to:
1. Calculate per-acre costs for each field
2. Compute bushels-per-dollar efficiency
3. Solve an optimization under a budget constraint

Without explicit reasoning steps, the model skips the constraint and gives a plausible-sounding but **wrong** answer.

---
## 3. Chain-of-Thought Prompting

**Chain-of-Thought (CoT)** prompting asks the model to show its reasoning step by step before giving a final answer. The seminal trick: append *"Let's think step by step"* to the prompt.

This simple change dramatically improves accuracy on math, logic, and multi-step tasks because it forces the model to allocate computation tokens to intermediate reasoning.

In [ ]:
# ── Chain-of-Thought prompt ──────────────────────────────────────────────────

cot_prompt = [
    {"role": "system", "content": "You are a precise analytical assistant. Always show your work."},
    {"role": "user", "content": HARD_PROBLEM + "\n\nLet's think step by step."},
]

SIMULATED_COT = """Let's think step by step.

Step 1: Calculate maintenance cost per acre for each field.
- Field A: $50/acre (base cost)
- Field B: $50 * 0.70 = $35/acre (30% less)
- Field C: $50 * 1.60 = $80/acre (60% more)

Step 2: Calculate bushels per dollar for each field.
- Field A: 240 / $50 = 4.80 bushels/$
- Field B: 180 / $35 = 5.14 bushels/$
- Field C: 300 / $80 = 3.75 bushels/$

Step 3: Rank fields by efficiency.
Best value: Field B (5.14), then Field A (4.80), then Field C (3.75).

Step 4: Allocate budget greedily, starting with the most efficient.
- Field B: 40 acres * $35 = $1,400. Remaining budget: $10,000 - $1,400 = $8,600.
- Field A: 40 acres * $50 = $2,000. Remaining budget: $8,600 - $2,000 = $6,600.
- Field C: $6,600 / $80 = 82.5 acres, but max is 40. So 40 acres * $80 = $3,200.
  Remaining budget: $6,600 - $3,200 = $3,400 (unused).

Wait -- we have leftover budget but all fields are at max capacity (40 acres each).

Step 5: Check total.
- Field B: 40 acres -> 40 * 180 = 7,200 bushels
- Field A: 40 acres -> 40 * 240 = 9,600 bushels
- Field C: 40 acres -> 40 * 300 = 12,000 bushels
- Total cost: $1,400 + $2,000 + $3,200 = $6,600 (within $10,000 budget)
- Total bushels: 28,800

Answer: Allocate 40 acres to each field. Total cost is $6,600 (under budget).
Total output: 28,800 bushels."""

cot_answer = call_llm(cot_prompt, SIMULATED_COT)
print_box("Chain-of-Thought Result", cot_answer)

In [ ]:
# ── Verify the CoT answer programmatically ───────────────────────────────────

cot_alloc = {"A": 40, "B": 40, "C": 40}
cot_cost = sum(cot_alloc[f] * costs[f] for f in cot_alloc)
cot_bushels = sum(cot_alloc[f] * yields[f] for f in cot_alloc)

print(f"CoT allocation cost:  ${cot_cost:,.0f}")
print(f"Budget:               ${budget:,}")
print(f"Within budget:        {cot_cost <= budget}")
print(f"Total bushels:        {cot_bushels:,}")
print(f"\nWith CoT, the model correctly computed per-acre costs, ranked")
print(f"by efficiency, and verified the budget constraint.")

In [ ]:
# ── Reusable CoT wrapper ─────────────────────────────────────────────────────

def chain_of_thought(problem: str, simulated: str = "") -> str:
    """Wrap any problem in a Chain-of-Thought prompt."""
    messages = [
        {"role": "system", "content": (
            "You are a precise analytical assistant. "
            "Break down every problem into numbered steps. "
            "Show all calculations. State your final answer clearly."
        )},
        {"role": "user", "content": problem + "\n\nLet's think step by step."},
    ]
    return call_llm(messages, simulated)


# Quick test with a simpler problem
simple_sim = """Step 1: A train travels 120 km in 1.5 hours.
Step 2: Speed = distance / time = 120 / 1.5 = 80 km/h.
Step 3: Time for 200 km at 80 km/h = 200 / 80 = 2.5 hours.

Answer: 2.5 hours."""

result = chain_of_thought(
    "A train travels 120 km in 1.5 hours. How long to travel 200 km at the same speed?",
    simulated=simple_sim
)
print(result)

### Why does CoT work?

1. **Token budget for reasoning** — Intermediate tokens give the model "scratch paper" to work through logic.
2. **Error visibility** — Mistakes in intermediate steps are visible and correctable.
3. **Decomposition** — Complex problems become a series of simpler sub-problems.

**Limitation:** CoT follows a single reasoning path. If that path starts wrong, it stays wrong. This motivates Tree-of-Thought.

---
## 4. Tree-of-Thought Reasoning

**Tree-of-Thought (ToT)** extends CoT by exploring **multiple reasoning paths** in parallel, evaluating each, and selecting the best one. Think of it as breadth-first search over reasoning steps.

The pattern:
1. **Propose** — Generate N different approaches to the problem.
2. **Evaluate** — Score each approach on correctness and completeness.
3. **Select** — Pick the best path (or vote across paths).

In [ ]:
# ── Tree-of-Thought: the problem ────────────────────────────────────────────

TOT_PROBLEM = """
A company needs to choose a cloud deployment strategy for an AI inference
service. The options are:
  (A) Single-region GPU cluster — lowest latency, highest risk of outage
  (B) Multi-region with load balancing — moderate latency, high availability
  (C) Edge deployment on CPUs — highest latency, lowest cost, best privacy

Requirements:
- 99.9% uptime SLA
- Must handle 10,000 requests/second at peak
- Budget is constrained (startup)
- Data contains PII subject to GDPR

Which option is best and why?
""".strip()

print(TOT_PROBLEM)

In [ ]:
# ── Step 1: Propose multiple reasoning paths ────────────────────────────────

def propose_thoughts(problem: str, n_thoughts: int = 3) -> list[str]:
    """Generate N independent reasoning paths for a problem."""
    thoughts = []
    for i in range(n_thoughts):
        messages = [
            {"role": "system", "content": (
                f"You are expert analyst #{i+1}. Reason independently about the "
                f"problem. Focus on a different angle than other analysts might. "
                f"Show your reasoning, then state your recommendation."
            )},
            {"role": "user", "content": problem + "\n\nThink step by step."},
        ]
        thought = call_llm(messages, SIMULATED_THOUGHTS[i], temperature=0.9)
        thoughts.append(thought)
    return thoughts


# Simulated expert reasoning paths
SIMULATED_THOUGHTS = [
    # Expert 1: focuses on uptime SLA
    """Approach: Prioritize the 99.9% uptime SLA requirement.

Step 1: 99.9% uptime means at most 8.76 hours downtime per year.
Step 2: Option A (single-region) has highest outage risk. A single region
  failure could easily exceed 8.76 hours. ELIMINATED.
Step 3: Option C (edge/CPU) can meet uptime if enough nodes exist, but
  10,000 req/s on CPUs requires massive horizontal scaling -- expensive.
Step 4: Option B (multi-region) inherently provides redundancy for 99.9%.

Recommendation: Option B.""",

    # Expert 2: focuses on GDPR and privacy
    """Approach: Prioritize GDPR compliance for PII data.

Step 1: GDPR requires data to stay within approved jurisdictions.
Step 2: Option C (edge) keeps data on-premise -- best for privacy.
Step 3: But 10,000 req/s on CPU edge nodes is very expensive.
Step 4: Option B (multi-region) can be GDPR-compliant if regions are
  within the EU. This is standard practice.
Step 5: Option A could also be EU-based, but fails the uptime SLA.

Recommendation: Option B (with EU-only regions).""",

    # Expert 3: focuses on cost
    """Approach: Prioritize budget constraints (startup).

Step 1: Option C is cheapest per node but needs many nodes for 10k req/s.
Step 2: GPU inference is ~10-50x faster than CPU. So Option C needs
  10-50x more hardware. Net cost may be HIGHER, not lower.
Step 3: Option A is cheapest GPU option but fails uptime SLA.
Step 4: Option B costs more than A but meets all requirements.
  Use spot/preemptible instances and autoscaling to control costs.

Recommendation: Option B (with cost optimization via autoscaling).""",
]

thoughts = propose_thoughts(TOT_PROBLEM, n_thoughts=3)
for i, t in enumerate(thoughts):
    print_box(f"Expert {i+1} Reasoning", t)

In [ ]:
# ── Step 2: Evaluate and vote ────────────────────────────────────────────────

def evaluate_thoughts(problem: str, thoughts: list[str]) -> dict:
    """Have an evaluator score each reasoning path and pick the best."""
    thought_summary = "\n\n".join(
        f"--- Expert {i+1} ---\n{t}" for i, t in enumerate(thoughts)
    )

    messages = [
        {"role": "system", "content": (
            "You are a senior evaluator. Review the expert analyses below. "
            "Score each on: correctness (0-10), completeness (0-10), "
            "practicality (0-10). Pick the best recommendation. "
            "Respond in JSON: {\"scores\": [{\"expert\": 1, \"correctness\": N, "
            "\"completeness\": N, \"practicality\": N},...], "
            "\"best_expert\": N, \"final_recommendation\": \"...\"}"
        )},
        {"role": "user", "content": f"Problem:\n{problem}\n\nExpert analyses:\n{thought_summary}"},
    ]

    SIMULATED_EVAL = json.dumps({
        "scores": [
            {"expert": 1, "correctness": 9, "completeness": 7, "practicality": 8},
            {"expert": 2, "correctness": 9, "completeness": 9, "practicality": 9},
            {"expert": 3, "correctness": 8, "completeness": 8, "practicality": 9},
        ],
        "best_expert": 2,
        "final_recommendation": (
            "Option B (multi-region with load balancing) using EU-only regions. "
            "This satisfies the 99.9% SLA through redundancy, handles 10k req/s "
            "with GPU inference, complies with GDPR by restricting to EU regions, "
            "and can be cost-optimized with autoscaling and spot instances."
        )
    }, indent=2)

    raw = call_llm(messages, SIMULATED_EVAL, temperature=0.3)
    return json.loads(raw)


evaluation = evaluate_thoughts(TOT_PROBLEM, thoughts)

print("Scores:")
for s in evaluation["scores"]:
    total = s["correctness"] + s["completeness"] + s["practicality"]
    print(f"  Expert {s['expert']}: correctness={s['correctness']}  "
          f"completeness={s['completeness']}  practicality={s['practicality']}  "
          f"total={total}")

print(f"\nBest expert: #{evaluation['best_expert']}")
print(f"\nFinal recommendation:")
print(textwrap.fill(evaluation["final_recommendation"], width=78))

In [ ]:
# ── Reusable Tree-of-Thought function ────────────────────────────────────────

def tree_of_thought(problem: str,
                    n_experts: int = 3,
                    simulated_thoughts: Optional[list[str]] = None,
                    simulated_eval: Optional[str] = None) -> dict:
    """Full Tree-of-Thought pipeline: propose, evaluate, select."""

    # Propose
    thoughts = []
    for i in range(n_experts):
        messages = [
            {"role": "system", "content": (
                f"You are expert analyst #{i+1}. Reason independently. "
                f"Show step-by-step reasoning, then state your recommendation."
            )},
            {"role": "user", "content": problem + "\n\nThink step by step."},
        ]
        sim = simulated_thoughts[i] if simulated_thoughts and i < len(simulated_thoughts) else "No simulation provided."
        thoughts.append(call_llm(messages, sim, temperature=0.9))

    # Evaluate
    thought_block = "\n\n".join(f"--- Expert {i+1} ---\n{t}" for i, t in enumerate(thoughts))
    eval_messages = [
        {"role": "system", "content": (
            "Evaluate the expert analyses. Score each (correctness, completeness, "
            "practicality, each 0-10). Return JSON with scores, best_expert, and "
            "final_recommendation."
        )},
        {"role": "user", "content": f"Problem:\n{problem}\n\n{thought_block}"},
    ]
    sim_eval = simulated_eval or '{"scores":[], "best_expert":1, "final_recommendation":"N/A"}'
    result = json.loads(call_llm(eval_messages, sim_eval, temperature=0.3))
    result["thoughts"] = thoughts
    return result


print("tree_of_thought() function defined.")
print("Usage: result = tree_of_thought('your problem here')")

### ToT vs CoT — When to use which?

| Aspect | CoT | ToT |
|--------|-----|-----|
| **API calls** | 1 | N proposals + 1 evaluation |
| **Cost** | Low | Higher |
| **Best for** | Well-defined problems | Ambiguous, multi-faceted decisions |
| **Failure mode** | Single bad reasoning path | Evaluator picks wrong path |

---
## 5. Self-Consistency

**Self-Consistency** (Wang et al., 2022) is a simpler alternative to full ToT:
1. Run the same CoT prompt **N times** with temperature > 0.
2. Extract the final answer from each run.
3. Take the **majority vote**.

This exploits the fact that correct reasoning paths are more likely than any single incorrect path, so they "outvote" errors.

In [ ]:
# ── Self-Consistency: the problem ────────────────────────────────────────────

SC_PROBLEM = """
A store sells apples in bags. A small bag has 6 apples and costs $4.
A large bag has 10 apples and costs $6. If you need exactly 28 apples
and want to minimize cost, how much will you spend?
""".strip()

print(SC_PROBLEM)

In [ ]:
# ── Generate multiple CoT samples ────────────────────────────────────────────

SIMULATED_SC_RESPONSES = [
    # Sample 1 — correct
    """Step 1: I need exactly 28 apples.
Step 2: Try combinations of small (6) and large (10) bags.
  - 0 large + ? small: 28/6 = 4.67 -- not exact.
  - 1 large (10) + ? small: 18/6 = 3 small. Cost: $6 + 3*$4 = $18.
  - 2 large (20) + ? small: 8/6 = 1.33 -- not exact.
  - 3 large (30) -- too many.
Step 3: Only valid combo: 1 large + 3 small = 10 + 18 = 28. Cost = $18.

Answer: $18""",

    # Sample 2 — correct
    """Let me enumerate options.
Large bags (L=10 apples, $6) and small bags (S=6 apples, $4).
10L + 6S = 28.
If L=1: 6S = 18, S=3. Cost = $6 + $12 = $18.
If L=0: 6S = 28. S=4.67. Not integer.
If L=2: 6S = 8. S=1.33. Not integer.
Only L=1, S=3 works.

Answer: $18""",

    # Sample 3 — wrong (arithmetic error)
    """Need 28 apples.
2 large bags = 20 apples ($12). Remaining = 8. 8/6 = 1.33.
Round up: 2 small bags = 12 apples. Total = 32 apples.
Cost = $12 + $8 = $20.

Answer: $20""",

    # Sample 4 — correct
    """I need exactly 28 apples. 10a + 6b = 28 where a,b >= 0 integers.
a=1, b=3: 10+18=28. Cost: 6+12=$18. Valid.
No other non-negative integer solutions exist.

Answer: $18""",

    # Sample 5 — wrong (misreads problem)
    """28 apples. Large bag is cheapest per apple ($0.60 vs $0.67).
Buy 3 large = 30 apples. Cost = $18. Close enough.

Answer: $18""",
]


def self_consistency(problem: str,
                     n_samples: int = 5,
                     simulated_responses: Optional[list[str]] = None) -> dict:
    """Run CoT n_samples times and take majority vote on the final answer."""
    responses = []
    answers = []

    for i in range(n_samples):
        messages = [
            {"role": "system", "content": "Solve the problem step by step. End with 'Answer: <value>'."},
            {"role": "user", "content": problem},
        ]
        sim = simulated_responses[i] if simulated_responses and i < len(simulated_responses) else ""
        response = call_llm(messages, sim, temperature=0.9)
        responses.append(response)

        # Extract the final answer after "Answer:"
        match = re.search(r"Answer:\s*(.+)", response)
        if match:
            answers.append(match.group(1).strip())

    # Majority vote
    vote_counts = Counter(answers)
    best_answer = vote_counts.most_common(1)[0] if vote_counts else ("No answer", 0)

    return {
        "responses": responses,
        "answers": answers,
        "vote_counts": dict(vote_counts),
        "majority_answer": best_answer[0],
        "majority_votes": best_answer[1],
        "total_samples": n_samples,
    }


result = self_consistency(SC_PROBLEM, n_samples=5, simulated_responses=SIMULATED_SC_RESPONSES)

print("Individual answers:", result["answers"])
print(f"Vote counts: {result['vote_counts']}")
print(f"\nMajority answer: {result['majority_answer']} ({result['majority_votes']}/{result['total_samples']} votes)")

In [ ]:
# ── Visualize the voting ─────────────────────────────────────────────────────

print("Self-Consistency Voting Results")
print("=" * 40)
for answer, count in sorted(result["vote_counts"].items(), key=lambda x: -x[1]):
    bar = "#" * (count * 8)
    marker = " <-- majority" if answer == result["majority_answer"] else ""
    print(f"  {answer:>6s}  |{bar}| {count} votes{marker}")
print()
print(f"Even though sample 3 had an arithmetic error and sample 5 changed")
print(f"the problem, the correct answer ($18) won the majority vote.")

### Self-Consistency takeaways

- **Cheap insurance:** Costs N API calls (all can run in parallel) but no evaluator call.
- **Works best when:** There is a single correct answer that can be extracted and compared.
- **Fails when:** The answer space is open-ended (essays, code) — hard to compare or vote.

---
## 6. Reflection and Self-Critique

**Reflection** is a pattern where the agent:
1. Generates an initial output.
2. Critiques its own output for errors, gaps, or improvements.
3. Revises based on the critique.
4. Optionally repeats until satisfied.

This mirrors how expert humans work: draft, review, revise.

In [ ]:
# ── Reflection: the task ─────────────────────────────────────────────────────

REFLECTION_TASK = """
Write a Python function that validates an email address.
It should check for:
- Presence of exactly one @ symbol
- Non-empty local part (before @)
- Domain has at least one dot
- No spaces anywhere
- Local part doesn't start or end with a dot
""".strip()

print(REFLECTION_TASK)

In [ ]:
# ── Step 1: Initial generation ───────────────────────────────────────────────

SIMULATED_DRAFT = '''\
```python
def validate_email(email: str) -> bool:
    if " " in email:
        return False
    if email.count("@") != 1:
        return False
    local, domain = email.split("@")
    if not local:
        return False
    if "." not in domain:
        return False
    return True
```
'''

generate_messages = [
    {"role": "system", "content": "You are a Python developer. Write clean, well-documented code."},
    {"role": "user", "content": REFLECTION_TASK},
]

draft = call_llm(generate_messages, SIMULATED_DRAFT)
print_box("Draft (v1)", draft)

In [ ]:
# ── Step 2: Self-critique ────────────────────────────────────────────────────

SIMULATED_CRITIQUE = """Issues found in the draft:

1. MISSING CHECK: The requirement says "local part doesn't start or end with
   a dot," but the code never checks for this. Emails like ".user@example.com"
   or "user.@example.com" would incorrectly pass.

2. MISSING CHECK: The domain part after the last dot should be at least 2
   characters (e.g., "user@example.c" is technically suspicious). While not
   explicitly required, it's a common validation.

3. EDGE CASE: Empty domain parts around dots are not checked.
   "user@.example.com" or "user@example..com" would pass.

4. NO DOCSTRING: The function lacks documentation.

5. NO TYPE HINTS on return aren't an issue (already bool), but adding
   examples in the docstring would help.

Priority fix: #1 (violates explicit requirement), then #3, then #4."""

critique_messages = [
    {"role": "system", "content": (
        "You are a senior code reviewer. Examine the code below against the "
        "original requirements. List every issue: bugs, missing requirements, "
        "edge cases, and code quality problems. Be thorough and specific."
    )},
    {"role": "user", "content": (
        f"Requirements:\n{REFLECTION_TASK}\n\nCode to review:\n{draft}"
    )},
]

critique = call_llm(critique_messages, SIMULATED_CRITIQUE, temperature=0.3)
print_box("Self-Critique", critique)

In [ ]:
# ── Step 3: Revise based on critique ─────────────────────────────────────────

SIMULATED_REVISION = '''\
```python
def validate_email(email: str) -> bool:
    """Validate an email address against basic structural rules.

    Checks:
        - Exactly one @ symbol
        - Non-empty local part that doesn't start/end with a dot
        - Domain has at least one dot, no empty segments
        - No spaces anywhere

    Examples:
        >>> validate_email("user@example.com")
        True
        >>> validate_email(".user@example.com")
        False
    """
    # No spaces allowed
    if " " in email:
        return False

    # Exactly one @
    if email.count("@") != 1:
        return False

    local, domain = email.split("@")

    # Non-empty local part
    if not local:
        return False

    # Local part must not start or end with a dot
    if local.startswith(".") or local.endswith("."):
        return False

    # Domain must have at least one dot
    if "." not in domain:
        return False

    # No empty domain segments (catches ".." and leading/trailing dots)
    domain_parts = domain.split(".")
    if any(len(part) == 0 for part in domain_parts):
        return False

    # TLD should be at least 2 characters
    if len(domain_parts[-1]) < 2:
        return False

    return True
```
'''

revise_messages = [
    {"role": "system", "content": (
        "You are a Python developer. Revise the code to address every issue "
        "identified in the review. Keep the fix minimal and clean."
    )},
    {"role": "user", "content": (
        f"Original requirements:\n{REFLECTION_TASK}\n\n"
        f"Current code:\n{draft}\n\n"
        f"Review feedback:\n{critique}\n\n"
        f"Please produce the revised code."
    )},
]

revision = call_llm(revise_messages, SIMULATED_REVISION)
print_box("Revised Code (v2)", revision)

In [ ]:
# ── Test the revised function ────────────────────────────────────────────────
# We paste the revised code here to verify it actually works.

def validate_email(email: str) -> bool:
    """Validate an email address against basic structural rules."""
    if " " in email:
        return False
    if email.count("@") != 1:
        return False
    local, domain = email.split("@")
    if not local:
        return False
    if local.startswith(".") or local.endswith("."):
        return False
    if "." not in domain:
        return False
    domain_parts = domain.split(".")
    if any(len(part) == 0 for part in domain_parts):
        return False
    if len(domain_parts[-1]) < 2:
        return False
    return True


# Test cases: (input, expected)
test_cases = [
    ("user@example.com", True),
    ("first.last@company.co.uk", True),
    ("user@localhost", False),           # no dot in domain
    (".user@example.com", False),        # local starts with dot
    ("user.@example.com", False),        # local ends with dot
    ("user@.example.com", False),        # empty domain segment
    ("user@example..com", False),        # double dot in domain
    ("user @example.com", False),        # space
    ("@example.com", False),             # empty local
    ("user@@example.com", False),        # double @
    ("user@example.c", False),           # TLD too short
    ("", False),                          # empty string
]

print(f"{'Input':<35} {'Expected':>8} {'Got':>8} {'Pass':>6}")
print("-" * 62)
all_pass = True
for email, expected in test_cases:
    got = validate_email(email)
    passed = got == expected
    all_pass = all_pass and passed
    status = "OK" if passed else "FAIL"
    print(f"{email:<35} {str(expected):>8} {str(got):>8} {status:>6}")

print(f"\nAll tests passed: {all_pass}")

In [ ]:
# ── Reusable reflection loop ─────────────────────────────────────────────────

def reflection_loop(task: str,
                    max_iterations: int = 3,
                    sim_drafts: Optional[list[str]] = None,
                    sim_critiques: Optional[list[str]] = None) -> dict:
    """Generate -> Critique -> Revise loop, up to max_iterations."""
    history = []

    # Initial generation
    gen_msg = [
        {"role": "system", "content": "You are an expert. Produce a high-quality response."},
        {"role": "user", "content": task},
    ]
    draft = call_llm(gen_msg, sim_drafts[0] if sim_drafts else "Initial draft.")
    history.append({"iteration": 0, "type": "draft", "content": draft})

    for i in range(max_iterations):
        # Critique
        crit_msg = [
            {"role": "system", "content": (
                "You are a critical reviewer. Find all issues with the response. "
                "If the response is excellent, say 'APPROVED' and nothing else."
            )},
            {"role": "user", "content": f"Task: {task}\n\nResponse to review:\n{draft}"},
        ]
        sim_c = sim_critiques[i] if sim_critiques and i < len(sim_critiques) else "APPROVED"
        critique = call_llm(crit_msg, sim_c, temperature=0.3)
        history.append({"iteration": i + 1, "type": "critique", "content": critique})

        if "APPROVED" in critique.upper():
            print(f"  Iteration {i+1}: APPROVED -- stopping early.")
            break

        print(f"  Iteration {i+1}: Issues found, revising...")

        # Revise
        rev_msg = [
            {"role": "system", "content": "Revise the response to address all critique points."},
            {"role": "user", "content": f"Task: {task}\n\nCurrent:\n{draft}\n\nCritique:\n{critique}"},
        ]
        sim_d = sim_drafts[i + 1] if sim_drafts and (i + 1) < len(sim_drafts) else draft
        draft = call_llm(rev_msg, sim_d)
        history.append({"iteration": i + 1, "type": "revision", "content": draft})

    return {"final": draft, "history": history, "iterations": len(history)}


# Quick demo
demo = reflection_loop(
    "Write a haiku about debugging.",
    max_iterations=2,
    sim_drafts=[
        "Bugs hide in the code\nPrint statements light the way\nFixed at 3 AM",
        "Silent bugs await\nStack traces guide the search home\nGreen tests bring the dawn",
    ],
    sim_critiques=[
        "The haiku is decent but 'Print statements light the way' is 7 syllables "
        "only if 'statements' is 2 syllables (it's 3). Syllable count: 5-8-5. Fix line 2.",
        "APPROVED. Syllable count is now 5-7-5. Good imagery."
    ]
)
print(f"\nFinal output:\n{demo['final']}")

### Reflection best practices

1. **Use a different persona** for the critic ("senior reviewer") than the generator.
2. **Set a stopping condition** — either "APPROVED" or a max iteration count.
3. **Pass the original requirements** to the critic so it checks against the spec, not just internal consistency.
4. **Temperature:** Use low temperature (0.2-0.3) for the critic, higher (0.7) for the generator.

---
## 7. Reasoning Improves Tool Selection

When an agent has access to multiple tools, reasoning before acting dramatically improves **which tool** gets selected. Without reasoning, the LLM pattern-matches on keywords. With reasoning, it considers the semantics of the request.

Below we simulate an agent with four tools and compare direct vs. reasoning-based tool selection.

In [ ]:
# ── Define a set of tools ────────────────────────────────────────────────────

TOOLS = {
    "web_search": {
        "description": "Search the web for current information. Best for facts, news, recent events.",
        "parameters": ["query"],
    },
    "calculator": {
        "description": "Evaluate mathematical expressions. Handles arithmetic, algebra, statistics.",
        "parameters": ["expression"],
    },
    "database_query": {
        "description": "Query the internal company database. Has employee records, sales data, inventory.",
        "parameters": ["sql"],
    },
    "document_search": {
        "description": "Semantic search over internal documents, policies, and knowledge base articles.",
        "parameters": ["query"],
    },
}

tool_descriptions = "\n".join(
    f"- {name}: {info['description']}" for name, info in TOOLS.items()
)
print("Available tools:")
print(tool_descriptions)

In [ ]:
# ── Test cases: ambiguous requests where keyword matching fails ───────────────

TOOL_TEST_CASES = [
    {
        "request": "What's our company's return policy?",
        "wrong_tool": "web_search",
        "right_tool": "document_search",
        "why": "'policy' might trigger web_search, but this is an internal document lookup.",
    },
    {
        "request": "How many employees joined in Q4 2025?",
        "wrong_tool": "calculator",
        "right_tool": "database_query",
        "why": "'how many' sounds like math, but this requires a COUNT query on the employee table.",
    },
    {
        "request": "What's the average of our last 12 monthly revenue figures?",
        "wrong_tool": "calculator",
        "right_tool": "database_query",
        "why": "'average' triggers calculator, but the data lives in the database.",
    },
    {
        "request": "Search for the latest news about our competitor's product launch.",
        "wrong_tool": "document_search",
        "right_tool": "web_search",
        "why": "'search' might trigger document_search, but competitor news is external.",
    },
]

for tc in TOOL_TEST_CASES:
    print(f"Request:    {tc['request']}")
    print(f"Naive pick: {tc['wrong_tool']} (keyword match)")
    print(f"Correct:    {tc['right_tool']}")
    print(f"Reason:     {tc['why']}")
    print()

In [ ]:
# ── Direct tool selection (no reasoning) ─────────────────────────────────────

def select_tool_direct(request: str, simulated: str = "") -> str:
    """Select a tool without explicit reasoning."""
    messages = [
        {"role": "system", "content": (
            f"You have these tools:\n{tool_descriptions}\n\n"
            f"Given the user request, respond with ONLY the tool name. "
            f"Nothing else."
        )},
        {"role": "user", "content": request},
    ]
    return call_llm(messages, simulated).strip()


# Simulate the naive (keyword-matching) responses
direct_results = []
for tc in TOOL_TEST_CASES:
    selected = select_tool_direct(tc["request"], simulated=tc["wrong_tool"])
    correct = selected == tc["right_tool"]
    direct_results.append(correct)
    status = "CORRECT" if correct else "WRONG"
    print(f"[{status}] '{tc['request'][:50]}...' -> {selected} (expected {tc['right_tool']})")

print(f"\nDirect accuracy: {sum(direct_results)}/{len(direct_results)}")

In [ ]:
# ── Reasoning-based tool selection ───────────────────────────────────────────

def select_tool_with_reasoning(request: str, simulated: str = "") -> dict:
    """Select a tool WITH explicit reasoning before choosing."""
    messages = [
        {"role": "system", "content": (
            f"You have these tools:\n{tool_descriptions}\n\n"
            f"Given the user request:\n"
            f"1. Analyze what the request actually needs (not just keywords).\n"
            f"2. Consider where the data lives (internal vs external, structured vs unstructured).\n"
            f"3. Pick the best tool.\n\n"
            f"Respond in JSON: {{\"reasoning\": \"...\", \"tool\": \"tool_name\"}}"
        )},
        {"role": "user", "content": request},
    ]
    raw = call_llm(messages, simulated, temperature=0.3)
    return json.loads(raw)


# Simulated reasoning responses
SIMULATED_REASONING = [
    json.dumps({"reasoning": "The user wants to know about 'our company's return policy.' This is an internal policy document, not something on the public web. The document_search tool searches internal knowledge base articles and policies.", "tool": "document_search"}),
    json.dumps({"reasoning": "The user wants a count of employees who joined in Q4 2025. This is structured data in the employee records. The database_query tool can run a SQL COUNT with a date filter.", "tool": "database_query"}),
    json.dumps({"reasoning": "The user wants an average of revenue figures. While 'average' sounds mathematical, the actual revenue data is stored in the database. We need database_query first to get the numbers, then the average can be computed in the SQL query itself.", "tool": "database_query"}),
    json.dumps({"reasoning": "The user wants news about a competitor's product launch. Competitor news is external, current information. The web_search tool is designed for recent external information.", "tool": "web_search"}),
]

reasoning_results = []
for i, tc in enumerate(TOOL_TEST_CASES):
    result = select_tool_with_reasoning(tc["request"], simulated=SIMULATED_REASONING[i])
    correct = result["tool"] == tc["right_tool"]
    reasoning_results.append(correct)
    status = "CORRECT" if correct else "WRONG"
    print(f"[{status}] '{tc['request'][:50]}...'")
    print(f"  Reasoning: {result['reasoning'][:80]}...")
    print(f"  Selected: {result['tool']}\n")

print(f"Reasoning accuracy: {sum(reasoning_results)}/{len(reasoning_results)}")
print(f"Direct accuracy:    {sum(direct_results)}/{len(direct_results)}")

**Key insight:** Adding a reasoning step before tool selection forces the model to consider:
- **Data location:** Is this internal or external? Structured or unstructured?
- **Operation type:** Am I retrieving data or computing on existing data?
- **Semantic intent:** What does the user *actually* need, beyond surface keywords?

This is why ReAct (Reason + Act) outperforms Act-only agents.

---
## 8. Choosing a Reasoning Strategy

With all these patterns available, how do you decide which to use? Below is a decision matrix and a helper function that recommends a strategy based on task characteristics.

In [ ]:
# ── Decision matrix ──────────────────────────────────────────────────────────

STRATEGY_MATRIX = {
    "direct": {
        "description": "Single LLM call, no reasoning scaffolding",
        "best_for": "Simple lookups, classification, formatting",
        "cost": "1 API call",
        "latency": "Low",
        "accuracy_boost": "None (baseline)",
    },
    "chain_of_thought": {
        "description": "Prompt model to show step-by-step reasoning",
        "best_for": "Math, logic, multi-step reasoning",
        "cost": "1 API call (more output tokens)",
        "latency": "Low-Medium",
        "accuracy_boost": "High for reasoning tasks",
    },
    "self_consistency": {
        "description": "Multiple CoT samples, majority vote",
        "best_for": "Tasks with a single correct answer (math, factual)",
        "cost": "N API calls (parallelizable)",
        "latency": "Medium (parallel) to High (sequential)",
        "accuracy_boost": "Very high for extractable answers",
    },
    "tree_of_thought": {
        "description": "Multiple reasoning paths + evaluator",
        "best_for": "Ambiguous problems, design decisions, trade-off analysis",
        "cost": "N+1 API calls",
        "latency": "High",
        "accuracy_boost": "High for complex decisions",
    },
    "reflection": {
        "description": "Generate -> Critique -> Revise loop",
        "best_for": "Code generation, writing, any task with verifiable quality criteria",
        "cost": "2-6 API calls per iteration",
        "latency": "High (sequential)",
        "accuracy_boost": "Very high for quality-sensitive tasks",
    },
}

# Pretty print
print(f"{'Strategy':<20} {'Cost':<25} {'Latency':<15} {'Best For'}")
print("-" * 90)
for name, info in STRATEGY_MATRIX.items():
    print(f"{name:<20} {info['cost']:<25} {info['latency']:<15} {info['best_for'][:40]}")

In [ ]:
# ── Strategy recommender function ────────────────────────────────────────────

def recommend_strategy(
    has_single_correct_answer: bool = False,
    requires_multi_step_reasoning: bool = False,
    is_ambiguous_or_subjective: bool = False,
    output_quality_critical: bool = False,
    latency_budget_ms: int = 5000,
    cost_sensitive: bool = True,
) -> list[dict]:
    """Recommend reasoning strategies based on task characteristics.

    Returns a ranked list of strategies with scores and reasoning.
    """
    candidates = []

    # Score each strategy
    for name, info in STRATEGY_MATRIX.items():
        score = 50  # baseline
        reasons = []

        if name == "direct":
            if not requires_multi_step_reasoning and not is_ambiguous_or_subjective:
                score += 30
                reasons.append("Simple task, no extra reasoning needed.")
            if cost_sensitive:
                score += 10
                reasons.append("Cheapest option.")
            if requires_multi_step_reasoning:
                score -= 40
                reasons.append("Will likely fail on multi-step reasoning.")

        elif name == "chain_of_thought":
            if requires_multi_step_reasoning:
                score += 30
                reasons.append("CoT excels at multi-step reasoning.")
            if cost_sensitive:
                score += 15
                reasons.append("Only 1 API call.")
            if latency_budget_ms < 2000:
                score += 10
                reasons.append("Low latency.")

        elif name == "self_consistency":
            if has_single_correct_answer:
                score += 35
                reasons.append("Perfect for tasks with one right answer.")
            if requires_multi_step_reasoning:
                score += 15
                reasons.append("Multiple CoT paths improve reliability.")
            if cost_sensitive:
                score -= 10
                reasons.append("Requires N parallel calls.")
            if is_ambiguous_or_subjective:
                score -= 20
                reasons.append("Hard to vote on open-ended answers.")

        elif name == "tree_of_thought":
            if is_ambiguous_or_subjective:
                score += 30
                reasons.append("Multiple perspectives help with ambiguity.")
            if not requires_multi_step_reasoning and not is_ambiguous_or_subjective:
                score -= 20
                reasons.append("Overkill for simple tasks.")
            if latency_budget_ms < 3000:
                score -= 15
                reasons.append("Too slow for tight latency budget.")

        elif name == "reflection":
            if output_quality_critical:
                score += 35
                reasons.append("Self-critique catches quality issues.")
            if cost_sensitive:
                score -= 15
                reasons.append("Multiple sequential calls are expensive.")
            if latency_budget_ms < 5000:
                score -= 15
                reasons.append("Sequential calls add latency.")

        candidates.append({
            "strategy": name,
            "score": score,
            "reasons": reasons,
            "description": info["description"],
        })

    # Sort by score descending
    candidates.sort(key=lambda x: x["score"], reverse=True)
    return candidates


print("recommend_strategy() function defined.")

In [ ]:
# ── Example 1: Math problem ──────────────────────────────────────────────────

print("Scenario: Solving a math word problem")
print("=" * 50)
recs = recommend_strategy(
    has_single_correct_answer=True,
    requires_multi_step_reasoning=True,
    is_ambiguous_or_subjective=False,
    output_quality_critical=False,
    latency_budget_ms=5000,
    cost_sensitive=True,
)
for r in recs:
    print(f"\n  {r['strategy']:<20} score={r['score']}")
    for reason in r["reasons"]:
        print(f"    - {reason}")

print(f"\n  >> Recommended: {recs[0]['strategy']}")

In [ ]:
# ── Example 2: Code generation for production ────────────────────────────────

print("Scenario: Generating production-quality code")
print("=" * 50)
recs = recommend_strategy(
    has_single_correct_answer=False,
    requires_multi_step_reasoning=True,
    is_ambiguous_or_subjective=False,
    output_quality_critical=True,
    latency_budget_ms=30000,
    cost_sensitive=False,
)
for r in recs:
    print(f"\n  {r['strategy']:<20} score={r['score']}")
    for reason in r["reasons"]:
        print(f"    - {reason}")

print(f"\n  >> Recommended: {recs[0]['strategy']}")

In [ ]:
# ── Example 3: Architecture decision ─────────────────────────────────────────

print("Scenario: Choosing a system architecture")
print("=" * 50)
recs = recommend_strategy(
    has_single_correct_answer=False,
    requires_multi_step_reasoning=True,
    is_ambiguous_or_subjective=True,
    output_quality_critical=True,
    latency_budget_ms=60000,
    cost_sensitive=False,
)
for r in recs:
    print(f"\n  {r['strategy']:<20} score={r['score']}")
    for reason in r["reasons"]:
        print(f"    - {reason}")

print(f"\n  >> Recommended: {recs[0]['strategy']}")

### Strategy selection guidelines

```
Is the task simple (lookup, classify, format)?
  YES -> Direct prompting
  NO  -> Does it need multi-step reasoning?
           YES -> Is there one correct answer?
                    YES -> Self-Consistency (or CoT if cost-sensitive)
                    NO  -> Is it ambiguous / subjective?
                             YES -> Tree-of-Thought
                             NO  -> Is output quality critical?
                                      YES -> Reflection
                                      NO  -> Chain-of-Thought
```

---
## 9. Exercises

### Exercise 1: Conceptual — When to Use Which Pattern

For each scenario below, decide which reasoning pattern is most appropriate and explain why.

| # | Scenario | Your Answer |
|---|----------|-------------|
| A | A chatbot that answers customer FAQs from a knowledge base | ? |
| B | An agent that writes and tests SQL queries for data analysts | ? |
| C | A medical triage system that must be highly accurate | ? |
| D | A brainstorming assistant for marketing campaign ideas | ? |
| E | An agent that solves competition-level math problems | ? |

In [ ]:
# ── Exercise 1: Use recommend_strategy() to check your answers ───────────────

# Example: Scenario A — customer FAQ chatbot
# Fill in the parameters based on the scenario characteristics,
# then see if the recommendation matches your intuition.

print("Scenario A: Customer FAQ chatbot")
recs_a = recommend_strategy(
    has_single_correct_answer=True,      # FAQs have known answers
    requires_multi_step_reasoning=False,  # simple lookup
    is_ambiguous_or_subjective=False,
    output_quality_critical=False,
    latency_budget_ms=2000,              # users expect fast responses
    cost_sensitive=True,                  # high volume
)
print(f"  Recommended: {recs_a[0]['strategy']}")

# TODO: Fill in scenarios B through E
# recs_b = recommend_strategy(...)
# recs_c = recommend_strategy(...)
# recs_d = recommend_strategy(...)
# recs_e = recommend_strategy(...)

### Exercise 2: Coding — Implement a Reflection Loop for Code Review

Build a reflection loop that:
1. Takes a Python function as input.
2. Generates a code review (bugs, style, performance).
3. Generates a revised version.
4. Reviews the revision.
5. Stops when the review says "APPROVED" or after 3 iterations.

Use the `reflection_loop()` function from Section 6 as a starting point, or build from scratch.

In [ ]:
# ── Exercise 2: Starter code ─────────────────────────────────────────────────

BUGGY_FUNCTION = '''
def find_duplicates(lst):
    dups = []
    for i in range(len(lst)):
        for j in range(len(lst)):
            if i != j and lst[i] == lst[j]:
                dups.append(lst[i])
    return dups
'''.strip()

print("Input function to review:")
print(BUGGY_FUNCTION)
print()
print("Known issues (do NOT look until you've tried):")
print("  1. O(n^2) time complexity")
print("  2. Returns duplicates multiple times")
print("  3. No type hints or docstring")
print()
print("TODO: Build a reflection loop that catches and fixes these issues.")

# Your code here:
# result = reflection_loop(
#     task=f"Review and improve this Python function:\n{BUGGY_FUNCTION}",
#     max_iterations=3,
#     sim_drafts=[...],     # provide simulated responses if USE_LIVE_API=False
#     sim_critiques=[...],
# )

### Exercise 3: Design — Build a Reasoning Strategy Selector Agent

Design (and optionally implement) an agent that:
1. Receives a user task.
2. Analyzes the task to determine its characteristics (single answer? ambiguous? quality-critical?).
3. Selects the best reasoning strategy.
4. Executes the task using that strategy.
5. Returns both the result and the strategy it chose (with justification).

This is a **meta-reasoning** agent — it reasons about how to reason.

In [ ]:
# ── Exercise 3: Starter code ─────────────────────────────────────────────────

def meta_reasoning_agent(task: str, simulated_analysis: Optional[str] = None) -> dict:
    """An agent that chooses HOW to reason before reasoning.

    Steps:
    1. Analyze the task to determine characteristics.
    2. Select a reasoning strategy.
    3. Execute using that strategy.
    4. Return result + justification.
    """

    # Step 1: Analyze the task
    analysis_prompt = [
        {"role": "system", "content": (
            "Analyze the following task and classify it. "
            "Respond in JSON with these boolean fields:\n"
            "  has_single_correct_answer\n"
            "  requires_multi_step_reasoning\n"
            "  is_ambiguous_or_subjective\n"
            "  output_quality_critical\n"
            "  estimated_latency_budget_ms (integer)\n"
            "  cost_sensitive (boolean)"
        )},
        {"role": "user", "content": task},
    ]

    default_analysis = json.dumps({
        "has_single_correct_answer": True,
        "requires_multi_step_reasoning": True,
        "is_ambiguous_or_subjective": False,
        "output_quality_critical": False,
        "estimated_latency_budget_ms": 5000,
        "cost_sensitive": True,
    })

    analysis_raw = call_llm(analysis_prompt, simulated_analysis or default_analysis)
    analysis = json.loads(analysis_raw)

    # Step 2: Select strategy using recommend_strategy()
    recs = recommend_strategy(
        has_single_correct_answer=analysis.get("has_single_correct_answer", False),
        requires_multi_step_reasoning=analysis.get("requires_multi_step_reasoning", False),
        is_ambiguous_or_subjective=analysis.get("is_ambiguous_or_subjective", False),
        output_quality_critical=analysis.get("output_quality_critical", False),
        latency_budget_ms=analysis.get("estimated_latency_budget_ms", 5000),
        cost_sensitive=analysis.get("cost_sensitive", True),
    )

    chosen = recs[0]

    # Step 3: TODO — Execute the task using the chosen strategy
    # This is where you would call chain_of_thought(), tree_of_thought(),
    # self_consistency(), or reflection_loop() based on chosen["strategy"].

    return {
        "task": task,
        "analysis": analysis,
        "chosen_strategy": chosen["strategy"],
        "score": chosen["score"],
        "reasons": chosen["reasons"],
        "result": "TODO: execute with chosen strategy",
    }


# Demo
demo = meta_reasoning_agent("What is 15% of 240 plus 8% of 350?")
print(f"Task:     {demo['task']}")
print(f"Analysis: {json.dumps(demo['analysis'], indent=2)}")
print(f"Strategy: {demo['chosen_strategy']} (score={demo['score']})")
print(f"Reasons:")
for r in demo["reasons"]:
    print(f"  - {r}")
print(f"\nTODO: Implement Step 3 to actually execute the task.")

---
## Summary

| Pattern | Key Idea | When to Use |
|---------|----------|-------------|
| **Direct** | Single call, no scaffolding | Simple tasks |
| **Chain-of-Thought** | "Let's think step by step" | Multi-step reasoning |
| **Tree-of-Thought** | Multiple paths + evaluator | Ambiguous decisions |
| **Self-Consistency** | N samples + majority vote | Single-answer tasks needing reliability |
| **Reflection** | Generate-critique-revise loop | Quality-critical outputs |

**The meta-lesson:** Reasoning is not free — it costs tokens, latency, and money. The best agents match their reasoning investment to the difficulty of the task. Simple tasks get simple strategies; hard tasks earn deeper reasoning.

In Chapter 6, we will build on these patterns to implement **memory** — giving agents the ability to learn and improve across conversations.